In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install transformers torch requests pandas --quiet

In [2]:
import os
AV_API_KEY = "YOUR_FREE_KEY_HERE"  
TOP50 = [
    "AAPL","MSFT","NVDA","AMZN","META","GOOGL","GOOG","BRK-B","LLY","JPM",
    "AVGO","XOM","TSLA","UNH","V","PG","MA","COST","HD","JNJ",
    "ABBV","MRK","WMT","BAC","CVX","NFLX","KO","CRM","AMD","PEP",
    "TMO","ORCL","LIN","ACN","MCD","CSCO","ABT","DHR","TXN","PM",
    "NKE","NEE","IBM","QCOM","LOW","CAT","RTX","SPGI","AMGN","GS"
]

DEVICE = "cuda" if __import__('torch').cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
import requests, time, json
import pandas as pd
from datetime import datetime, timedelta

def fetch_av_news(ticker: str, api_key: str, limit: int = 200) -> pd.DataFrame:
   
    url = (
        f"https://www.alphavantage.co/query"
        f"?function=NEWS_SENTIMENT&tickers={ticker}"
        f"&limit={limit}&apikey={api_key}"
    )
    r = requests.get(url, timeout=15)
    data = r.json()

    if 'feed' not in data:
        print(f"  WARNING {ticker}: {data.get('Note', data.get('Information', 'unknown error'))}")
        return pd.DataFrame()

    records = []
    for item in data['feed']:
       
        ticker_sentiment = next(
            (ts for ts in item.get('ticker_sentiment', []) if ts['ticker'] == ticker),
            None
        )
        if ticker_sentiment is None:
            continue

        records.append({
            'ticker'           : ticker,
            'date'             : item['time_published'][:8],   # YYYYMMDD
            'headline'         : item['title'],
            'summary'          : item.get('summary', ''),
            'av_sentiment'     : item.get('overall_sentiment_label', ''),
            'av_score'         : float(item.get('overall_sentiment_score', 0)),
            'ticker_relevance' : float(ticker_sentiment.get('relevance_score', 0)),
            'ticker_sentiment' : ticker_sentiment.get('sentiment_label', ''),
            'ticker_score'     : float(ticker_sentiment.get('ticker_sentiment_score', 0)),
        })

    return pd.DataFrame(records)

def fetch_all_tickers(tickers, api_key, delay=2.5):
    """Fetch news for all tickers with rate limiting."""
    all_frames = []
    for i, t in enumerate(tickers):
        df = fetch_av_news(t, api_key)
        if not df.empty:
            all_frames.append(df)
        print(f"  [{i+1}/{len(tickers)}] {t}: {len(df)} articles")
        time.sleep(delay)   

    combined = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
    return combined

print("Fetching news from Alpha Vantage (takes ~3 min for 50 tickers)...")
news_raw = fetch_all_tickers(TOP50, AV_API_KEY)
news_raw.to_csv('/kaggle/working/news_raw.csv', index=False)
print(f"Total articles: {len(news_raw):,}")
news_raw.head()

Fetching news from Alpha Vantage (takes ~3 min for 50 tickers)...
  [1/50] AAPL: 50 articles
  [2/50] MSFT: 50 articles
  [3/50] NVDA: 50 articles
  [4/50] AMZN: 50 articles
  [5/50] META: 50 articles
  [6/50] GOOGL: 50 articles
  [7/50] GOOG: 50 articles
  [8/50] BRK-B: 50 articles
  [9/50] LLY: 50 articles
  [10/50] JPM: 50 articles
  [11/50] AVGO: 50 articles
  [12/50] XOM: 50 articles
  [13/50] TSLA: 50 articles
  [14/50] UNH: 50 articles
  [15/50] V: 50 articles
  [16/50] PG: 50 articles
  [17/50] MA: 50 articles
  [18/50] COST: 50 articles
  [19/50] HD: 50 articles
  [20/50] JNJ: 50 articles
  [21/50] ABBV: 50 articles
  [22/50] MRK: 50 articles
  [23/50] WMT: 50 articles
  [24/50] BAC: 50 articles
  [25/50] CVX: 50 articles
  WARNING NFLX: We have detected your API key as YOUR_FREE_KEY_HERE and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.
  [26/5

,ticker,date,headline,summary,av_sentiment,av_score,ticker_relevance,ticker_sentiment,ticker_score
0,AAPL,20260327,"Apple adds Bosch, Cirrus Logic, others to US m...",Apple is expanding its American Manufacturing ...,Somewhat-Bullish,0.286377,0.986893,,0.309978
1,AAPL,20260326,"Bernstein Lowers Qualcomm (QCOM) PT, Says Expe...",Bernstein Research downgraded QUALCOMM Incorpo...,Somewhat-Bearish,-0.299505,0.708663,,-0.223151
2,AAPL,20260326,RIP Mac Pro: Apple officially kills its tower ...,Apple has officially discontinued the Mac Pro ...,Neutral,0.108768,0.976088,,0.129692
3,AAPL,20260326,Sonos Inc Stock: Premium Wireless Audio Leader...,"Sonos Inc, a leader in premium wireless audio,...",Neutral,0.067073,0.728174,,-0.127408
4,AAPL,20260326,"Bernstein Lowers Qualcomm (QCOM) PT, Says Expe...",Bernstein Research downgraded QUALCOMM Incorpo...,Somewhat-Bearish,-0.304470,0.741285,,-0.222699


In [5]:
import torch
import pandas as pd  
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FINBERT_MODEL = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
finbert = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL).to(DEVICE)
finbert.eval()

def finbert_batch(texts: list, batch_size: int = 32) -> np.ndarray:
    """Returns (N, 3) array of [prob_neg, prob_neu, prob_pos]."""
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors='pt'
        ).to(DEVICE)
        with torch.no_grad():
            logits = finbert(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)


news_raw = pd.read_csv('/kaggle/working/news_raw.csv')


texts = (news_raw['headline'] + '. ' + news_raw['summary'].str[:100].fillna('')).tolist()

print(f"Running FinBERT on {len(texts):,} articles...")
probs = finbert_batch(texts)


news_raw['finbert_neg'] = probs[:, 0]
news_raw['finbert_neu'] = probs[:, 1]
news_raw['finbert_pos'] = probs[:, 2]
news_raw['finbert_score'] = probs[:, 2] - probs[:, 0]  # pos - neg composite



labels = np.array(['negative', 'neutral', 'positive'])
news_raw['finbert_label'] = labels[probs.argmax(axis=1)]


print(news_raw['finbert_label'].value_counts())
news_raw.to_csv('/kaggle/working/news_scored.csv', index=False)
news_raw[['ticker','date','headline','finbert_label','finbert_score']].head(10)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running FinBERT on 1,250 articles...
finbert_label
negative    494
positive    417
neutral     339
Name: count, dtype: int64


,ticker,date,headline,finbert_label,finbert_score
0,AAPL,20260327,"Apple adds Bosch, Cirrus Logic, others to US m...",negative,-0.556717
1,AAPL,20260326,"Bernstein Lowers Qualcomm (QCOM) PT, Says Expe...",neutral,0.066687
2,AAPL,20260326,RIP Mac Pro: Apple officially kills its tower ...,neutral,0.107389
3,AAPL,20260326,Sonos Inc Stock: Premium Wireless Audio Leader...,negative,-0.298899
4,AAPL,20260326,"Bernstein Lowers Qualcomm (QCOM) PT, Says Expe...",neutral,0.123249
5,AAPL,20260326,Semilux Plummets 22%: The Shocking Collapse of...,neutral,0.006961
6,AAPL,20260326,"Vanguard disaggregates holdings, reports 0 AAP...",positive,0.839613
7,AAPL,20260326,Chip Shortage 2026: Why CPUs From Intel And AM...,neutral,0.008980
8,AAPL,20260326,Apple may open up Siri to AI assistants such a...,positive,0.635638
9,AAPL,20260326,Cirrus Logic Shares up 7.2% After Apple Adds,negative,-0.906828


In [6]:
news_scored = pd.read_csv('/kaggle/working/news_scored.csv')
news_scored['date'] = pd.to_datetime(news_scored['date'], format='%Y%m%d')

def aggregate_daily_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate multiple articles per ticker/day into 4 features:
    - weighted mean finbert_score (weight = ticker_relevance)
    - article count
    - sentiment momentum (today's score - 5-day rolling mean)
    - score std dev (high std = conflicting signals = uncertainty)
    """
    agg = df.groupby(['ticker','date']).apply(lambda g: pd.Series({
        'sent_score'  : np.average(g['finbert_score'], weights=g['ticker_relevance'].clip(0.01)),
        'article_count': len(g),
        'sent_std'    : g['finbert_score'].std() if len(g) > 1 else 0.0,
        'pos_ratio'   : (g['finbert_label'] == 'positive').mean(),
    })).reset_index()

   
    agg = agg.sort_values(['ticker','date'])
    agg['sent_momentum'] = agg.groupby('ticker')['sent_score'].transform(
        lambda x: x - x.rolling(5, min_periods=1).mean().shift(1)
    )
    return agg

daily_sent = aggregate_daily_sentiment(news_scored)
daily_sent.to_csv('/kaggle/working/daily_sentiment.csv', index=False)
print(f"Daily sentiment rows: {len(daily_sent):,}")
print(daily_sent.describe())

Daily sentiment rows: 120
                      date  sent_score  article_count    sent_std   pos_ratio  \
count                  120  120.000000     120.000000  120.000000  120.000000   
mean   2026-03-13 06:24:00    0.080124      10.416667    0.418378    0.408467   
min    2026-01-02 00:00:00   -0.927641       1.000000    0.000000    0.000000   
25%    2026-03-16 18:00:00   -0.193135       1.000000    0.000000    0.116013   
50%    2026-03-24 00:00:00    0.004391       6.000000    0.583590    0.336667   
75%    2026-03-26 00:00:00    0.337093      15.250000    0.683428    0.600000   
max    2026-03-27 00:00:00    0.926207      50.000000    1.083559    1.000000   
std                    NaN    0.472979      11.868030    0.330509    0.346100   

       sent_momentum  
count      95.000000  
mean        0.004662  
min        -1.342896  
25%        -0.293339  
50%         0.005902  
75%         0.342705  
max         1.314516  
std         0.549295  


/tmp/ipykernel_55/1115666824.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = df.groupby(['ticker','date']).apply(lambda g: pd.Series({
